In [1]:
import pandas as pd
train = pd.read_csv('../data/train.csv', sep=",")
test = pd.read_csv('../data/test.csv')

train_ID = train['Id']
test_ID = test['Id']

train.drop('Id', axis = 1, inplace = True)
test.drop('Id', axis = 1, inplace = True)

SalePrice = train['SalePrice']
train.drop('SalePrice', axis=1, inplace = True)

### this is causing problems:
Depending on the version of scikit-learn, the two variables containing years might cause failures in Lightgbm

In [2]:
train = train.drop(['YearBuilt', 'YearRemodAdd'], axis=1)

### preprocessing steps:

In [3]:
categorical = [var for var in train.columns if train[var].dtype=='O']
numerical = [var for var in train.columns if train[var].dtype!='O']

### preprocessing steps as done with elasticnet
For comparability, we first do the same preprocessing steps as were done in the house-prices linear-regression notebook

Perhaps, we should have a look in the [documentation](https://lightgbm.readthedocs.io/en/latest/Parameters.html)

In [4]:
import lightgbm
model = lightgbm.LGBMRegressor(boosting_type='gbdt', num_leaves=31, max_depth=-1, learning_rate=0.1, 
                                n_estimators=500, subsample_for_bin=20000, objective='regression', 
                                subsample=1.0, subsample_freq=0, colsample_bytree=1.0, 
                                n_jobs=- 1, silent=False, importance_type='split',
                                scale_pos_weight = 1.0,
                                reg_alpha = 0.05, # L1
                                reg_lambda = 0.05)  # L2
# an alias for the boosting_type is 'gbrt' = gradient boosted regression trees

Please note the use of a pipeline together with cross-validation. The cross-validation step is there, to ensure, that we have a good estimate of the algorithms performance. However, when we do the transformation steps on the whole data-set, information will leak from into the test-set: <br>
 - The mean for missing-values imputation should be computed only on the training data and then be applied to the test-data
 - The best parameters for the box-cox transformation should not be found with the test-set included
 
__We need an evaluation of the model as if it would already be employed in a production process.__

The __pipeline__ takes care of the right modeling order. It has the same methods as all sklearn models:<br>
 - .fit() to fit the pipeline to data
 - .fit_transform() to fit the pipline to data and do a transform (also training the model)
 - .transform() to only apply the fitted pipeline
 
The __cross_val_score__ function splits the data in cv-folds. It applies .fit_transform() to the test-data folds and evaluates the test-fold with only the .transform() methods. 

In [5]:
import sys
import os
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
# box-cox needs strictly positive values:

train[numerical] += 1

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='mean')),
    ('scaler', PowerTransformer(method='box-cox'))])

categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)#, sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical),
        ('cat', categorical_transformer, categorical),
        ])

clf = cross_val_score(
        make_pipeline(
            preprocessor,
            #InteractionsTransformer(),
            model
            ),
         train, np.log1p(SalePrice), scoring = 'neg_mean_squared_error'
)

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000560 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3062
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 193
[LightGBM] [Info] Start training from score 12.021409
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [6]:
np.mean(np.sqrt(-clf))

0.13350852041292444

### preprocessing steps in the lightgbm-way
Now, we evaluate the performace of the lightgbm model without any preprocessing. Lightgbm, however, needs to know which variables are considered to be categorical.<br>
In the cross-validation, we stick to the use of a pipeline.

In [7]:
import lightgbm
model = lightgbm.LGBMRegressor(boosting_type='gbdt', num_leaves=31, max_depth=- 1, learning_rate=0.1, 
                                n_estimators=500, subsample_for_bin=20000, objective='regression', 
                                subsample=1.0, subsample_freq=0, colsample_bytree=1.0, 
                                n_jobs=- 1, silent=False, importance_type='split',
                                scale_pos_weight = 1.0,
                                reg_alpha = 0.05, # L1
                                reg_lambda = 0.05,  # L2
                                categorical_feature = categorical)

In [7]:
numeric_features = numerical
categorical_features = categorical

for cat_feat in categorical_features:
    train[cat_feat] = train[cat_feat].astype('category')
    
clf = cross_val_score(
        make_pipeline(
            model
            ),
         train, np.log1p(SalePrice), scoring = 'neg_mean_squared_error'
)

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000216 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3129
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 72
[LightGBM] [Info] Start training from score 12.021409
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000170 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3139
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 73
[LightGBM] [Info] Start training from score 12.020737
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [8]:
np.mean(np.sqrt(-clf))

0.13116592835353336

As you can see, the result is even slightly better as the. one obtained with preprocessing. As we discussed, trees do not rely on the distributional form of the variables, i.e., there's no need for the box-cox transform. Missing values are treated as an extra category of the variable, there's no need for missing value imputation.

### Tuning Gradient-Boosting-Models
Preprocessing is less intensive for gradient-boosing models. But these models come with a lot of different parameters. Finding the best combination of these parameters is often the most compute-intensive step when fine-tuning these models.<br>
Again, we use a pipeline. The class __GridSearchCV__ takes this pipeline together with a parameter grid. Then it does a cross-validation search over all possible combinations of the given parameters.

In [9]:
clf = GridSearchCV(
        make_pipeline(
            model
            ),
            param_grid={'lgbmregressor__max_depth': [3, 4, 5],
                        'lgbmregressor__n_estimators' : [900, 1000, 1100],
                        'lgbmregressor__learning_rate': 10**(np.linspace(-2, 0, 3)),
                        'lgbmregressor__subsample': [0.8],
                        'lgbmregressor__reg_alpha': [0.0],
                        'lgbmregressor__reg_lambda': [0.1]
},
         cv=5, refit=False, scoring = 'neg_mean_squared_error'
)
clf.fit(train, np.log1p(SalePrice))

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000278 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3129
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 72
[LightGBM] [Info] Start training from score 12.021409
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


... (249 outputs left out)

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/lightgbm/sklearn.py:598: UserWarning: 'silent' argument is deprecated and will be removed in a future release of LightGBM. Pass 'verbose' parameter via keyword arguments instead.
  _log_warning("'silent' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('lgbmregressor',
                                        LGBMRegressor(n_estimators=500,
                                                      objective='regression',
                                                      reg_alpha=0.05,
                                                      reg_lambda=0.05,
                                                      scale_pos_weight=1.0,
                                                      silent=False,
                                                      subsample_for_bin=20000))]),
             param_grid={'lgbmregressor__learning_rate': array([0.01, 0.1 , 1.  ]),
                         'lgbmregressor__max_depth': [3, 4, 5],
                         'lgbmregressor__n_estimators': [900, 1000, 1100],
                         'lgbmregressor__reg_alpha': [0.0],
                         'lgbmregressor__reg_lambda': [0.1],
                         'lgbmregressor__subsample': [0.8]},
             refit=False, scoring='neg_mean_squared_error')

In [10]:
#clf.grid_scores_, 
clf.best_params_, np.sqrt(-clf.best_score_)

({'lgbmregressor__learning_rate': 0.01,
  'lgbmregressor__max_depth': 4,
  'lgbmregressor__n_estimators': 1100,
  'lgbmregressor__reg_alpha': 0.0,
  'lgbmregressor__reg_lambda': 0.1,
  'lgbmregressor__subsample': 0.8},
 0.1271716459991531)

## How to correctly evaluate new ideas

In one of the courses, a student had the following idea:<br>
 - since Trees have some desirable properties and since GBT are built greedily (the first trees contribute most to the result), we could combine trees with elastic-net (ridge-regression)
 - trees can model the most important interactions among the most important variables
 - why not add the prediction of the trees of a GBT to the variables of  an elastic-net model?

### the GBM of sklearn allows for single-tree predictions
We discussed this in the first session: the leaf-information of the single trees of a GBM could be interesting categorical features for linear classifiers. For the example with elasticnet, we choose the most important variables and added their interaction terms to the other variables. The first trees of a GBM should automatically contain the most important variable splits and interactions.


Contrary to lightgbm, the GradientBoostingRegressor of sklean needs some preprocessing steps:

In [14]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import FunctionTransformer
model = GradientBoostingRegressor(learning_rate = 0.01, max_depth=4, n_estimators=100, subsample=0.8)

train[numerical] = train[numerical].astype('float32')
numeric_transformer = Pipeline(steps=[
   ('imputer', SimpleImputer(missing_values=np.nan, strategy='median'))])
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)      
pipeline = make_pipeline(
            ColumnTransformer(transformers=[('num', numeric_transformer, numerical), 
                                          ('empty', categorical_transformer, categorical)]),
            model
            )
pipeline.fit(train, np.log1p(SalePrice))

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['MSSubClass', 'LotFrontage',
                                                   'LotArea', 'OverallQual',
                                                   'OverallCond', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                                   '1stFlrSF', '2ndFlrSF',
                                                   'LowQualFinSF', 'GrLivArea',
                                                   'BsmtFullBath',
                                                   'BsmtHalfBath', '...
                                                   'Condition2', 'BldgType',
                                                   'HouseStyle', 'RoofStyle',
                                                   'RoofMatl', 'Exterior1st',
                                                   'Exterior2nd', 'MasVnrType',
                                                   'ExterQual', 'ExterCond',
                                                   'Foundation', 'BsmtQual',
                                                   'BsmtCond', 'BsmtExposure',
                                                   'BsmtFinType1',
                                                   'BsmtFinType2', 'Heating',
                                                   'HeatingQC', 'CentralAir',
                                                   'Electrical', ...])])),
                ('gradientboostingregressor',
                 GradientBoostingRegressor(learning_rate=0.01, max_depth=4,
                                           subsample=0.8))])

Now, we can add the single-tree-predictions of a GradientBoostingRegressor to the training data:

In [15]:
tree_features = []
for j in range(0, 15):
    train['gbm_feature_' + str(j)]= pipeline['gradientboostingregressor'].estimators_[j][0].predict(pipeline['columntransformer'].fit_transform(train))
    train['gbm_feature_' + str(j)] = train['gbm_feature_' + str(j)].astype('category')
    tree_features.append('gbm_feature_' + str(j))


In [16]:
train.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond',
       'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
       'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF',
       '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath',
       'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF',
       'En

In [20]:
categorical = categorical + tree_features

This is what the single tree predictions look like: For the firest tree, they are predicted residuals with respect to the mean of the log-transformed Sale-Price.

In [ ]:
train[['gbm_feature_0']]

We can run a best-parameter grid-search for the elastic net, with the tree-preditions in the training data:

In [22]:
import sys
import os
import numpy as np
from sklearn.linear_model import ElasticNet
sys.path.append(os.path.abspath('../scripts'))
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = numerical
categorical_features = categorical

train.replace([np.inf, -np.inf], np.nan, inplace=True)

numeric_transformer = Pipeline(steps=[
('imputer', SimpleImputer(missing_values=np.nan, strategy='mean')),
('scaler', PowerTransformer(method='box-cox'))]) 
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical),
        ('cat', categorical_transformer, categorical),
        ])

clf = GridSearchCV(
        make_pipeline(
            preprocessor,
            ElasticNet()
            ),
            param_grid={'elasticnet__alpha': 10**(np.linspace(-1, 0.2, 5)),
                        'elasticnet__l1_ratio': np.linspace(0, 1, 6)},
         cv=5, refit=True, scoring = 'neg_mean_squared_error'
)


In [23]:
train[numerical] += 1
clf.fit(train, np.log1p(SalePrice))

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.014e+01, tolerance: 1.891e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.155e+01, tolerance: 1.837e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/pyth

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.979e+01, tolerance: 1.891e-02 Linear regression models with null weight for the l1 regularizatio

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.556e+01, tolerance: 1.837e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/martin

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          PowerTransformer())]),
                                                                         ['MSSubClass',
                                                                          'LotFrontage',
                                                                          'LotArea',
                                                                          'OverallQual',
                                                                          'OverallCond',
                                                                          'MasVnrArea',
                                                                          'BsmtFinSF1',
                                                                          'BsmtFinSF2',
                                                                          'BsmtUnfSF',
                                                                          'TotalBsmtSF',
                                                                          '1stFlrSF',
                                                                          '2ndFlrSF',
                                                                          'LowQualFinSF',
                                                                          'G...
                                                                          'ExterCond',
                                                                          'Foundation',
                                                                          'BsmtQual',
                                                                          'BsmtCond',
                                                                          'BsmtExposure',
                                                                          'BsmtFinType1',
                                                                          'BsmtFinType2',
                                                                          'Heating',
                                                                          'HeatingQC',
                                                                          'CentralAir',
                                                                          'Electrical', ...])])),
                                       ('elasticnet', ElasticNet())]),
             param_grid={'elasticnet__alpha': array([0.1       , 0.19952623, 0.39810717, 0.79432823, 1.58489319]),
                         'elasticnet__l1_ratio': array([0. , 0.2, 0.4, 0.6, 0.8, 1. ])},
             scoring='neg_mean_squared_error')

In [24]:
display(np.sqrt(-clf.best_score_), clf.best_params_)

0.12697177618345834

{'elasticnet__alpha': 0.1, 'elasticnet__l1_ratio': 0.0}

### BUT:<br>
we are violating the rule, that training-data and test-data should be independent. Because the trees were built with all data - not only the training-parts.

### this is an interesting exercise for our brains: what is happening here???
  - can we do stacking? Stacking could come to the rescue: Train the trees on the 4 folds of the training-data and predict for the fifth fold. The problem here is the fact that trees are build on a random subset of the variables, such that predictions of the single trees might differe for the different combinations of the training-folds.
  - what might be the effect of the learning-rate in the gbm? The higher the learning rate, the more aggressively the tress try to greedily predict the residuals. Higher learning rates might lead faster to overfitting.
  - can we say, the gbm breaks non-linearities in continuous variables and is modelling interactions? Yes, trees can break non-linearities and can model interactions.
  - or: is the linear model applying individual weights to each tree - instead of one learning rate for the whole tree? After fitting the individual trees sequentially (gbm), is the linear model re-weighing the single-leaf predictions in one step, all together?

### Since we know, that it's forbidden what we do, we could do the following:
  - split the whole data-set into 5 parts (sklearn.model_selection.StratifiedKFold)
  - train 4 parts of the data with the GradientBoostedRegressor
  - apply the fitted GradientBoostedRegressor to the train-set (4 parts) as well as to the test-set (1 part)
  - make parameter-search for elastic-net in the train-set (4 parts) and apply to the test-set (1 part)
  - collect results in a list
  - repeat 5 times and report average result
  - perhaps repeat and change the learning-rate for the GradientBoostedRegressor until we find (no more) overfitting

In [1]:
import pandas as pd
train = pd.read_csv('../data/train.csv', sep=",")
test = pd.read_csv('../data/test.csv')

train_ID = train['Id']
test_ID = test['Id']

train.drop('Id', axis = 1, inplace = True)
test.drop('Id', axis = 1, inplace = True)

SalePrice = train['SalePrice']
train.drop('SalePrice', axis=1, inplace = True)


numerical = [var for var in train.columns if train[var].dtype!='O']

In [26]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5)
skf.get_n_splits(train, SalePrice)



performance = []
for train_index, test_index in skf.split(train, SalePrice):
    
    categorical = [var for var in train.columns if train[var].dtype=='O']
    train_X = train.loc[train_index, :]
    test_X =train.loc[test_index, :]
    train_y = SalePrice[train_index]
    test_y = SalePrice[test_index]
    pipeline.fit(train_X, np.log1p(train_y))
    
    tree_features = []
    for j in range(0, 15):
        train_X['gbm_feature_' + str(j)]= pipeline['gradientboostingregressor'].estimators_[j][0].predict(pipeline['columntransformer'].fit_transform(train_X))
        train_X['gbm_feature_' + str(j)] = train_X['gbm_feature_' + str(j)].astype('category')
        test_X['gbm_feature_' + str(j)]= pipeline['gradientboostingregressor'].estimators_[j][0].predict(pipeline['columntransformer'].transform(test_X))
        test_X['gbm_feature_' + str(j)] = test_X['gbm_feature_' + str(j)].astype('category')
        tree_features.append('gbm_feature_' + str(j))
        
    categorical = categorical + tree_features
    
    numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='mean')),
    ('scaler', PowerTransformer(method='box-cox')) # 'yeo-johnson'
    ])

    categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical),
            ])

    clf = GridSearchCV(
            make_pipeline(
                preprocessor,
                ElasticNet()
                ),
                param_grid={'elasticnet__alpha': 10**(np.linspace(-2, 0.2, 5)),
                            'elasticnet__l1_ratio': np.linspace(0, 1, 6)},
             cv=5, refit=True, scoring = 'neg_mean_squared_error'
    )
    train_X[numerical] += 1
    test_X[numerical] += 1
    clf.fit(train_X, np.log1p(train_y))
    performance.append(np.sqrt(np.mean((clf.predict(test_X)-np.log1p(test_y))**2)))




/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.478e+00, tolerance: 1.429e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, c

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.200e+01, tolerance: 1.429e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.134e+01, tolerance: 1.428e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted usi

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.718e+00, tolerance: 1.547e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.628e+00, tolerance: 1.516e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted usi

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.068e+01, tolerance: 1.547e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.976e+01, tolerance: 1.516e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted usi

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.134e+00, tolerance: 1.424e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.657e+00, tolerance: 1.497e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted usi

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.936e+01, tolerance: 1.424e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.941e+01, tolerance: 1.497e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted usi

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.617e+00, tolerance: 1.479e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.104e+00, tolerance: 1.548e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted usi

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.421e+00, tolerance: 1.443e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.793e+00, tolerance: 1.444e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted usi

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.189e+01, tolerance: 1.443e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.113e+01, tolerance: 1.444e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted usi

In [27]:
performance



[0.12760116185684778,
 0.1357029715519423,
 0.11414718017263298,
 0.14073983021166553,
 0.12919042304903378]

In [28]:
np.mean(performance)

0.12947631336842447

what seemed to be a good idea turns out to be __overfitting__: results are worse then in the 'forbidden' trial.

### EBM = explainable boosting machine

In [2]:
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from sklearn.model_selection import cross_val_score
set_visualize_provider(InlineProvider())

In [3]:
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import numpy as np

In [4]:
ebm = ExplainableBoostingRegressor()
clf = cross_val_score(
        ebm,
         train, np.log1p(SalePrice), scoring = 'neg_mean_squared_error'
)


/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/interpret/glassbox/_ebm/_ebm.py:723: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/interpret/glassbox/_ebm/_ebm.py:723: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(
/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/interpret/glassbox/_ebm/_ebm.py:723: UserWarning

In [5]:
np.mean(np.sqrt(-clf))

0.12349501308273676

In [6]:
y_mean = np.mean(np.log1p(SalePrice))
new_y = np.log1p(SalePrice)- y_mean
ebm.fit(train, new_y)

/home/martin/miniconda3/envs/imbalanced/lib/python3.9/site-packages/interpret/glassbox/_ebm/_ebm.py:723: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(


ExplainableBoostingRegressor()

# Ames Housing Dataset Variable Descriptions

### General Information
1. **MSSubClass**: Identifies the type of dwelling involved in the sale.
   - Example: 20 = 1-STORY 1946 & NEWER ALL STYLES.

2. **MSZoning**: Identifies the general zoning classification of the sale.
   - Example: Residential Low Density (RL), Commercial (C), etc.

3. **LotFrontage**: Linear feet of street connected to the property.

4. **LotArea**: Lot size in square feet.

5. **Street**: Type of road access to the property.
   - Example: Paved (Pave), Gravel (Grvl).

6. **Alley**: Type of alley access to the property.
   - Example: Paved (Pave), Gravel (Grvl), None (NA).

7. **LotShape**: General shape of the property.
   - Example: Regular (Reg), Slightly irregular (IR1), etc.

8. **LandContour**: Flatness of the property.
   - Example: Level (Lvl), Banked (Bank), etc.

9. **Utilities**: Type of utilities available.
   - Example: All public utilities (AllPub), Electricity and gas only (NoSewr), etc.

10. **LotConfig**: Lot configuration.
    - Example: Inside lot (Inside), Corner lot (Corner), etc.

11. **LandSlope**: Slope of the property.
    - Example: Gentle slope (Gtl), Moderate slope (Mod), etc.

12. **Neighborhood**: Physical location within Ames city limits.
    - Example: Bloomington Heights (Blmngtn), Old Town (OldTown), etc.

13. **Condition1**: Proximity to various conditions (e.g., main road, railroad).
    - Example: Adjacent to arterial street (Artery), Adjacent to feeder street (Feedr), etc.

14. **Condition2**: Proximity to various conditions (if more than one is present).

15. **BldgType**: Type of dwelling.
    - Example: Single-family detached (1Fam), Townhouse end unit (TwnhsE), etc.

16. **HouseStyle**: Style of dwelling.
    - Example: One story (1Story), Two story (2Story), etc.

### Quality and Condition
17. **OverallQual**: Rates the overall material and finish of the house (1-10 scale).

18. **OverallCond**: Rates the overall condition of the house (1-10 scale).

### Exterior Features
19. **RoofStyle**: Type of roof.
    - Example: Flat (Flat), Gable (Gable), etc.

20. **RoofMatl**: Roof material.
    - Example: Standard shingle (CompShg), Wood shingles (WdShngl), etc.

21. **Exterior1st**: Exterior covering on the house.
    - Example: Asphalt shingles (AsbShng), Brick common (BrkComm), etc.

22. **Exterior2nd**: Exterior covering on the house (if more than one material).

23. **MasVnrType**: Masonry veneer type.
    - Example: Brick face (BrkFace), Stone (Stone), None (None).

24. **MasVnrArea**: Masonry veneer area in square feet.

25. **ExterQual**: Evaluates the quality of the exterior material.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

26. **ExterCond**: Evaluates the present condition of the exterior material.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

### Basement Features
27. **Foundation**: Type of foundation.
    - Example: Poured concrete (PConc), Cinder block (CBlock), etc.

28. **BsmtQual**: Evaluates the height of the basement.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

29. **BsmtCond**: Evaluates the general condition of the basement.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

30. **BsmtExposure**: Refers to walkout or garden-level walls.
    - Example: Good exposure (Gd), Average exposure (Av), etc.

31. **BsmtFinType1**: Rating of the basement finished area.
    - Example: Good living quarters (GLQ), Average living quarters (ALQ), etc.

32. **BsmtFinSF1**: Type 1 finished square feet.

33. **BsmtFinType2**: Rating of the basement finished area (if multiple types).

34. **BsmtFinSF2**: Type 2 finished square feet.

35. **BsmtUnfSF**: Unfinished square feet of the basement area.

36. **TotalBsmtSF**: Total square feet of the basement area.

### Heating and Cooling
37. **Heating**: Type of heating.
    - Example: Gas forced warm air furnace (GasA), Electric furnace (Elec), etc.

38. **HeatingQC**: Heating quality and condition.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

39. **CentralAir**: Central air conditioning.
    - Example: Yes (Y), No (N).

40. **Electrical**: Electrical system.
    - Example: Standard circuit breakers (SBrkr), Fuse box (FuseA), etc.

### Floor Area
41. **1stFlrSF**: First-floor square feet.

42. **2ndFlrSF**: Second-floor square feet.

43. **LowQualFinSF**: Low-quality finished square feet (all floors).

44. **GrLivArea**: Above-grade (ground) living area square feet.

### Bathrooms
45. **BsmtFullBath**: Basement full bathrooms.

46. **BsmtHalfBath**: Basement half bathrooms.

47. **FullBath**: Full bathrooms above grade.

48. **HalfBath**: Half bathrooms above grade.

### Bedrooms and Kitchens
49. **BedroomAbvGr**: Bedrooms above grade.

50. **KitchenAbvGr**: Kitchens above grade.

51. **KitchenQual**: Kitchen quality.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

### Rooms and Functionality
52. **TotRmsAbvGrd**: Total rooms above grade (does not include bathrooms).

53. **Functional**: Home functionality rating.
    - Example: Typical functionality (Typ), Minor deductions (Min1), etc.

### Fireplaces
54. **Fireplaces**: Number of fireplaces.

55. **FireplaceQu**: Fireplace quality.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

### Garage Features
56. **GarageType**: Garage location.
    - Example: Attached to the home (Attchd), Detached from the home (Detchd), etc.

57. **GarageYrBlt**: Year the garage was built.

58. **GarageFinish**: Interior finish of the garage.
    - Example: Finished (Fin), Rough-finished (RFn), Unfinished (Unf).

59. **GarageCars**: Size of the garage in car capacity.

60. **GarageArea**: Size of the garage in square feet.

61. **GarageQual**: Garage quality.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

62. **GarageCond**: Garage condition.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

### Driveway and Porches
63. **PavedDrive**: Paved driveway.
    - Example: Paved (Y), Partial pavement (P), Dirt/Gravel (N).

64. **WoodDeckSF**: Wood deck area in square feet.

65. **OpenPorchSF**: Open porch area in square feet.

66. **EnclosedPorch**: Enclosed porch area in square feet.

67. **3SsnPorch**: Three-season porch area in square feet.

68. **ScreenPorch**: Screen porch area in square feet.

### Pool and Misc Features
69. **PoolArea**: Pool area in square feet.

70. **PoolQC**: Pool quality.
    - Example: Excellent (Ex), Good (Gd), Average (TA), etc.

71. **Fence**: Fence quality.
    - Example: Good privacy (GdPrv), Minimum privacy (MnPrv), etc.

72. **MiscFeature**: Miscellaneous feature not covered in other categories.
    - Example: Shed (Shed), Second garage (Gar2), etc.

73. **MiscVal**: Value of miscellaneous feature.

### Sale Information
74. **MoSold**: Month sold.

75. **YrSold**: Year sold.

76. **SaleType**: Type of sale.
    - Example: Warranty deed (WD), Court officer deed (CWD), etc.

77. **SaleCondition**: Condition of sale.
    - Example: Normal sale (Normal), Abnormally low price (Abnorml), etc.

In [ ]:
show(ebm.explain_global())

In [ ]:
show(ebm.explain_local(train[:5], new_y[:5]), 0)

In [9]:
train[0:1]

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal


In [11]:
new_y[0]

0.22364172145415395